## Explicación

## Pipeline


.mrxs + H&E.xml
    → espécimen H&E
    → tintas H&E
    → H&E en coordenadas del espécimen

.hdr + HSI_ink_annotation.png
    → espécimen HSI
    → tintas HSI
    → HSI en coordenadas del espécimen

comparar tintas + contornos
    → elegir orientación H&E
    → alinear H&E con HSI
    → devolver imágenes/resultados

## Código

### Funciones de otros archivos

2_Segmentacion_HSI

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from spectral import open_image


def extract_specimen_only_from_hsi_path(
    hdr_path,
    r_nm=650,
    g_nm=550,
    b_nm=450,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=False
):
    """
    Carga una imagen hiperespectral ENVI desde una ruta .hdr y devuelve una pseudo-RGB
    donde solo se ve el espécimen; el resto queda negro.

    Parámetros
    ----------
    hdr_path : str
        Ruta al archivo .hdr de la imagen hiperespectral.

    r_nm, g_nm, b_nm : float
        Longitudes de onda usadas para construir la pseudo-RGB.

    grow_pixels : int
        Número de píxeles para agrandar ligeramente el contorno del espécimen.
        Útil para evitar cortar tejido.

    show_original : bool
        Si True, plotea la pseudo-RGB original.

    show_result : bool
        Si True, plotea la imagen con solo el espécimen.

    return_mask : bool
        Si True, devuelve también la máscara binaria del espécimen.

    Returns
    -------
    specimen_only_rgb : np.ndarray
        Imagen RGB uint8 con solo el espécimen visible y el resto negro.

    Opcionalmente:
    mask_specimen : np.ndarray
        Máscara uint8 del espécimen, con valores 0 y 255.
    """

    # ============================================================
    # 1. Cargar cubo HSI
    # ============================================================
    img = open_image(hdr_path)
    cube = np.asarray(img.load(), dtype=np.float32)

    wavelengths = np.array(
        [float(w) for w in img.metadata["wavelength"]],
        dtype=np.float32
    )

    # ============================================================
    # 2. Funciones internas
    # ============================================================
    def find_nearest_band(wavelengths, target_nm):
        wavelengths_arr = np.asarray(wavelengths, dtype=float)
        idx = np.argmin(np.abs(wavelengths_arr - target_nm))
        return idx

    def robust_normalize(channel, p_low=2, p_high=98):
        ch = channel.astype(np.float32)

        lo = np.percentile(ch, p_low)
        hi = np.percentile(ch, p_high)

        if hi <= lo:
            return np.zeros_like(ch, dtype=np.float32)

        ch = (ch - lo) / (hi - lo)
        ch = np.clip(ch, 0, 1)

        return ch

    # ============================================================
    # 3. Crear pseudo-RGB
    # ============================================================
    r_idx = find_nearest_band(wavelengths, r_nm)
    g_idx = find_nearest_band(wavelengths, g_nm)
    b_idx = find_nearest_band(wavelengths, b_nm)

    r = robust_normalize(cube[:, :, r_idx])
    g = robust_normalize(cube[:, :, g_idx])
    b = robust_normalize(cube[:, :, b_idx])

    pseudo_rgb = np.stack([r, g, b], axis=-1)
    rgb_u8 = (np.clip(pseudo_rgb, 0, 1) * 255).astype(np.uint8)

    # ============================================================
    # 4. Detectar caja azul
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    R = rgb_u8[:, :, 0].astype(np.int16)
    G = rgb_u8[:, :, 1].astype(np.int16)
    B = rgb_u8[:, :, 2].astype(np.int16)

    V = hsv[:, :, 2]

    lower_blue = np.array([85, 10, 40], dtype=np.uint8)
    upper_blue = np.array([125, 180, 255], dtype=np.uint8)

    mask_hsv = cv2.inRange(hsv, lower_blue, upper_blue)

    mask_dominance = (
        ((B - R) > 8) &
        ((G - R) > 3) &
        (V > 50)
    )

    mask_blue = mask_hsv & (mask_dominance.astype(np.uint8) * 255)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))

    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)
    mask_blue = cv2.dilate(mask_blue, kernel_dilate, iterations=1)

    if np.count_nonzero(mask_blue) == 0:
        raise ValueError("No se detectó la caja azul. Revisa los umbrales HSV.")

    # ============================================================
    # 5. Quedarse con el componente azul más grande
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_blue,
        connectivity=8
    )

    if num_labels <= 1:
        raise ValueError("No se encontró ningún componente azul válido.")

    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_label = 1 + np.argmax(areas)

    mask_box_blue = (labels == largest_label).astype(np.uint8) * 255

    # ============================================================
    # 6. Contorno interno de la caja azul = espécimen
    # ============================================================
    contours, hierarchy = cv2.findContours(
        mask_box_blue,
        cv2.RETR_CCOMP,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if hierarchy is None:
        raise ValueError("No se encontraron contornos en la máscara de la caja azul.")

    hierarchy = hierarchy[0]

    inner_contours = [
        cnt for cnt, h in zip(contours, hierarchy)
        if h[3] != -1
    ]

    if len(inner_contours) == 0:
        raise ValueError("No se encontró ningún hueco interno en la caja azul.")

    specimen_contour = max(inner_contours, key=cv2.contourArea)

    mask_specimen = np.zeros_like(mask_box_blue, dtype=np.uint8)
    cv2.drawContours(
        mask_specimen,
        [specimen_contour],
        -1,
        255,
        thickness=cv2.FILLED
    )

    # Suavizar pequeñas irregularidades
    kernel_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_specimen = cv2.morphologyEx(
        mask_specimen,
        cv2.MORPH_CLOSE,
        kernel_smooth
    )

    # Agrandar ligeramente el contorno
    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )

        mask_specimen = cv2.dilate(
            mask_specimen,
            kernel_grow,
            iterations=1
        )

    # ============================================================
    # 7. Aplicar máscara: espécimen visible, resto negro
    # ============================================================
    specimen_only_rgb = rgb_u8.copy()
    specimen_only_rgb[mask_specimen == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if show_original and show_result:
        plt.figure(figsize=(12, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(specimen_only_rgb)
        plt.title("Solo espécimen")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(specimen_only_rgb)
        plt.title("Solo espécimen")
        plt.axis("off")
        plt.show()

    if return_mask:
        return specimen_only_rgb, mask_specimen

    return specimen_only_rgb

3_Segmentacion_H&E

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import openslide


def extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    sat_thresh=8,
    val_thresh=253,
    od_thresh=0.025,
    min_area=500,
    open_kernel_size=3,
    close_kernel_size=45,
    grow_pixels=8,
    use_convex_hull=False,
    show_original=True,
    show_result=True,
    debug=False,
    return_mask=False,
    return_info=False
):
    """
    Extrae el tejido de una H&E tipo .mrxs usando el contorno externo del tejido.

    Por defecto devuelve solo:
        tissue_only_rgb

    Opcionalmente puede devolver:
        mask_tissue
        info

    Parameters
    ----------
    slide_path : str or Path
        Ruta al archivo .mrxs.

    max_dim : int
        Tamaño máximo del lado largo usado para leer la imagen a baja resolución.

    sat_thresh : int
        Umbral de saturación HSV. Más bajo detecta tejido más pálido.

    val_thresh : int
        Umbral de brillo HSV. Más alto permite zonas más claras.

    od_thresh : float
        Umbral de optical density. Más bajo detecta tejido muy claro.

    min_area : int
        Área mínima de componentes a conservar.

    open_kernel_size : int
        Kernel para eliminar ruido pequeño.

    close_kernel_size : int
        Kernel para cerrar huecos y unir zonas del tejido.

    grow_pixels : int
        Dilatación final del contorno para no cortar tejido.

    use_convex_hull : bool
        Si True, usa convex hull. Es más conservador, puede meter más fondo.

    show_original : bool
        Plotea la H&E original leída.

    show_result : bool
        Plotea el resultado final.

    debug : bool
        Muestra pasos intermedios.

    return_mask : bool
        Si True, devuelve también la máscara final.

    return_info : bool
        Si True, devuelve también información auxiliar.
    """

    slide_path = Path(slide_path)

    if not slide_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {slide_path}")

    # ============================================================
    # 1. Cargar slide a baja resolución
    # ============================================================
    slide = openslide.OpenSlide(str(slide_path))
    level_dims = slide.level_dimensions

    chosen_level = None
    for i, (w, h) in enumerate(level_dims):
        if max(w, h) <= max_dim:
            chosen_level = i
            break

    if chosen_level is None:
        chosen_level = len(level_dims) - 1

    level_w, level_h = level_dims[chosen_level]

    rgb_pil = slide.read_region(
        location=(0, 0),
        level=chosen_level,
        size=(level_w, level_h)
    ).convert("RGB")

    slide.close()

    rgb_u8 = np.array(rgb_pil, dtype=np.uint8)

    # ============================================================
    # 2. Mapas HSV y optical density
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    rgb_float = rgb_u8.astype(np.float32)
    od = -np.log((rgb_float + 1.0) / 255.0)
    od_sum = od.sum(axis=2)

    # ============================================================
    # 3. Máscara inicial: tejido teñido o tejido pálido
    # ============================================================
    mask_sat = (
        (S > sat_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_od = (
        (od_sum > od_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_raw = cv2.bitwise_or(mask_sat, mask_od)

    # ============================================================
    # 4. Morfología
    # ============================================================
    mask_morph = mask_raw.copy()

    if open_kernel_size > 0:
        kernel_open = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (open_kernel_size, open_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_OPEN, kernel_open)

    if close_kernel_size > 0:
        kernel_close = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (close_kernel_size, close_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_CLOSE, kernel_close)

    # ============================================================
    # 5. Componentes conectados
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_morph,
        connectivity=8
    )

    mask_components = np.zeros_like(mask_morph, dtype=np.uint8)

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area >= min_area:
            mask_components[labels == label_id] = 255

    if np.count_nonzero(mask_components) == 0:
        raise ValueError(
            "No se detectó tejido. Prueba sat_thresh=5, od_thresh=0.012, val_thresh=255."
        )

    # ============================================================
    # 6. Extraer contorno externo y rellenarlo
    # ============================================================
    contours, _ = cv2.findContours(
        mask_components,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        raise ValueError("No se encontraron contornos de tejido.")

    # Normalmente el tejido principal es el mayor contorno
    largest_contour = max(contours, key=cv2.contourArea)

    mask_contour = np.zeros_like(mask_components, dtype=np.uint8)

    if use_convex_hull:
        hull = cv2.convexHull(largest_contour)
        cv2.drawContours(mask_contour, [hull], -1, 255, thickness=cv2.FILLED)
    else:
        cv2.drawContours(mask_contour, [largest_contour], -1, 255, thickness=cv2.FILLED)

    # Dilatar ligeramente para no cortar borde
    mask_final = mask_contour.copy()

    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )
        mask_final = cv2.dilate(mask_final, kernel_grow, iterations=1)

    # ============================================================
    # 7. Aplicar máscara: tejido visible, resto negro
    # ============================================================
    tissue_only_rgb = rgb_u8.copy()
    tissue_only_rgb[mask_final == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if debug:
        od_vis = (od_sum - od_sum.min()) / (od_sum.max() - od_sum.min() + 1e-8)

        plt.figure(figsize=(18, 10))

        plt.subplot(2, 4, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(2, 4, 2)
        plt.imshow(S, cmap="gray")
        plt.title("Saturación HSV")
        plt.axis("off")

        plt.subplot(2, 4, 3)
        plt.imshow(od_vis, cmap="gray")
        plt.title("Optical density")
        plt.axis("off")

        plt.subplot(2, 4, 4)
        plt.imshow(mask_raw, cmap="gray")
        plt.title("Máscara inicial")
        plt.axis("off")

        plt.subplot(2, 4, 5)
        plt.imshow(mask_morph, cmap="gray")
        plt.title("Después morfología")
        plt.axis("off")

        plt.subplot(2, 4, 6)
        plt.imshow(mask_components, cmap="gray")
        plt.title("Componentes filtrados")
        plt.axis("off")

        plt.subplot(2, 4, 7)
        plt.imshow(mask_final, cmap="gray")
        plt.title("Máscara final por contorno")
        plt.axis("off")

        plt.subplot(2, 4, 8)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original and show_result:
        plt.figure(figsize=(10, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")
        plt.show()

    # ============================================================
    # 9. Return limpio
    # ============================================================
    outputs = [tissue_only_rgb]

    if return_mask:
        outputs.append(mask_final)

    if return_info:
        info = {
            "chosen_level": chosen_level,
            "read_dimensions": (level_w, level_h),
            "sat_thresh": sat_thresh,
            "val_thresh": val_thresh,
            "od_thresh": od_thresh,
            "grow_pixels": grow_pixels,
            "use_convex_hull": use_convex_hull,
            "mask_area_px": int(np.count_nonzero(mask_final)),
        }
        outputs.append(info)

    if len(outputs) == 1:
        return tissue_only_rgb

    return tuple(outputs)

### Resto de código

In [ ]:
import re
import cv2
import difflib
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle


# ============================================================
# 1. Utilidades generales
# ============================================================

def _read_rgb_image_safe(path):
    """
    Lee una imagen RGB de forma robusta en Windows.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo:\n{path}")

    data = np.fromfile(str(path), dtype=np.uint8)
    img_bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)

    if img_bgr is None:
        raise ValueError(f"OpenCV no pudo leer la imagen:\n{path}")

    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def _get_ink_color_rgb(ink):
    if ink == "green":
        return (0, 255, 0)
    if ink == "red":
        return (255, 0, 0)
    if ink == "blue":
        return (0, 0, 255)
    return (255, 255, 0)


def _get_ink_color_mpl(ink):
    if ink == "green":
        return "lime"
    if ink == "red":
        return "red"
    if ink == "blue":
        return "blue"
    return "yellow"


def _normalize_text(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = s.encode("ascii", "ignore").decode("ascii")
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return s.strip()


def _canonical_ink_name(name):
    """
    Convierte nombres tipo Greeen, RED, verde, etc. a:
        green / red / blue
    """
    aliases = {
        "green": ["green", "greeen", "verde"],
        "red": ["red", "rojo", "roja"],
        "blue": ["blue", "azul"],
    }

    n = _normalize_text(name)
    tokens = n.split()

    for canonical, alias_list in aliases.items():
        for alias in alias_list:
            alias_n = _normalize_text(alias)

            if alias_n == n or alias_n in tokens:
                return canonical

            if len(tokens) == 1:
                ratio = difflib.SequenceMatcher(None, tokens[0], alias_n).ratio()
                if ratio > 0.78:
                    return canonical

    return None


def _polygon_features(points):
    pts = np.asarray(points, dtype=np.float64)

    x = pts[:, 0]
    y = pts[:, 1]

    bbox = (
        float(x.min()),
        float(y.min()),
        float(x.max()),
        float(y.max()),
    )

    centroid = pts.mean(axis=0)

    if len(pts) >= 3:
        area = abs(cv2.contourArea(pts.astype(np.float32)))
    else:
        area = 0.0

    return {
        "points": pts,
        "centroid": centroid,
        "bbox": bbox,
        "area": area,
    }


def _bbox_from_mask(mask, margin=20):
    """
    Bounding box de una máscara binaria.
    """
    mask_bool = mask > 0
    ys, xs = np.where(mask_bool)

    if len(xs) == 0:
        raise ValueError("La máscara está vacía.")

    h, w = mask_bool.shape

    x1 = max(int(xs.min()) - margin, 0)
    y1 = max(int(ys.min()) - margin, 0)
    x2 = min(int(xs.max()) + margin, w - 1)
    y2 = min(int(ys.max()) + margin, h - 1)

    return x1, y1, x2, y2


def _binary_bbox(mask):
    mask = mask > 0
    ys, xs = np.where(mask)

    if len(xs) == 0:
        raise ValueError("La máscara está vacía.")

    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def _bbox_center_and_size(bbox):
    x1, y1, x2, y2 = bbox

    w = x2 - x1 + 1
    h = y2 - y1 + 1

    cx = x1 + w / 2.0
    cy = y1 + h / 2.0

    return np.array([cx, cy], dtype=float), np.array([w, h], dtype=float)


def _transform_points_affine(points, M):
    pts = np.asarray(points, dtype=float)

    if pts.ndim == 1:
        pts = pts.reshape(1, 2)

    pts_h = np.concatenate(
        [pts, np.ones((len(pts), 1), dtype=float)],
        axis=1
    )

    return pts_h @ M.T


def _affine_2x3_to_3x3(M):
    M3 = np.eye(3, dtype=np.float64)
    M3[:2, :] = M
    return M3


def _affine_3x3_to_2x3(M3):
    return M3[:2, :].astype(np.float32)


# ============================================================
# 2. Extraer anotaciones H&E XML + HSI PNG
# ============================================================

def _read_he_xml_ink_annotations(he_xml_path, keep_inks=("green", "red", "blue")):
    """
    Lee H&E.xml y extrae solo anotaciones de tintas.
    """
    he_xml_path = Path(he_xml_path)
    root = ET.parse(he_xml_path).getroot()

    source_roi = root.find(".//source/region_of_interest")
    destination_roi = root.find(".//destination/region_of_interest")

    source_size = None
    destination_size = None

    if source_roi is not None:
        source_size = (
            int(float(source_roi.attrib["width"])),
            int(float(source_roi.attrib["height"])),
        )

    if destination_roi is not None:
        destination_size = (
            int(float(destination_roi.attrib["width"])),
            int(float(destination_roi.attrib["height"])),
        )

    he_annotations = []

    for ann in root.findall(".//annotation"):
        raw_name = ann.attrib.get("name", "")
        ink = _canonical_ink_name(raw_name)

        if ink is None or ink not in keep_inks:
            continue

        points = []

        for p in ann.findall(".//p"):
            x = float(p.attrib["x"])
            y = float(p.attrib["y"])
            points.append((x, y))

        if len(points) == 0:
            continue

        features = _polygon_features(points)

        he_annotations.append({
            "modality": "HE",
            "ink": ink,
            "raw_name": raw_name,
            "type": ann.attrib.get("type", ""),
            "color_bgr_xml": ann.attrib.get("color_bgr", ""),
            **features,
        })

    # Inferir si las coordenadas usan source o destination.
    coord_size = source_size
    coord_system = "source"

    if len(he_annotations) > 0 and destination_size is not None:
        all_points = np.concatenate([a["points"] for a in he_annotations], axis=0)
        max_x = all_points[:, 0].max()
        max_y = all_points[:, 1].max()

        if max_x <= destination_size[0] and max_y <= destination_size[1]:
            coord_size = destination_size
            coord_system = "destination"

    meta = {
        "xml_path": str(he_xml_path),
        "source_size": source_size,
        "destination_size": destination_size,
        "coord_system_inferred": coord_system,
        "coord_size_used": coord_size,
    }

    return he_annotations, meta


def _mask_hsi_annotation_color(img_rgb, ink):
    """
    Segmenta los rectángulos artificiales de tinta del PNG anotado.
    """
    r = img_rgb[:, :, 0].astype(np.int16)
    g = img_rgb[:, :, 1].astype(np.int16)
    b = img_rgb[:, :, 2].astype(np.int16)

    if ink == "green":
        mask = (g >= 120) & (g - r >= 70) & (g - b >= 35)
    elif ink == "red":
        mask = (r >= 150) & (r - g >= 70) & (r - b >= 70)
    elif ink == "blue":
        mask = (b >= 150) & (b - r >= 60) & (b - g >= 40)
    else:
        raise ValueError(f"Tinta no soportada: {ink}")

    return mask.astype(np.uint8) * 255


def _detect_hsi_png_ink_annotations(hsi_ink_png_path, keep_inks=("green", "red", "blue")):
    """
    Detecta rectángulos/cuadrados de tinta dibujados en el PNG HSI.
    """
    img_rgb = _read_rgb_image_safe(hsi_ink_png_path)
    H, W = img_rgb.shape[:2]

    hsi_annotations = []
    debug_masks = {}

    for ink in keep_inks:
        mask = _mask_hsi_annotation_color(img_rgb, ink)

        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        mask_clean = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

        debug_masks[ink] = mask_clean

        n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
            mask_clean,
            connectivity=8
        )

        candidates = []

        for i in range(1, n_labels):
            x, y, w, h, area = stats[i]

            if area < 80:
                continue

            if w < 10 or h < 10:
                continue

            fill_frac = area / float(w * h)

            # Un rectángulo dibujado como borde tiene poca ocupación en su bbox.
            if fill_frac > 0.45:
                continue

            candidates.append({
                "bbox": (int(x), int(y), int(x + w - 1), int(y + h - 1)),
                "centroid": np.array(centroids[i], dtype=float),
                "area": int(area),
                "fill_frac": float(fill_frac),
            })

        # Nos quedamos con el candidato más grande por tinta.
        if len(candidates) == 0:
            continue

        c = max(candidates, key=lambda d: d["area"])

        x1, y1, x2, y2 = c["bbox"]

        rect_points = np.array(
            [
                [x1, y1],
                [x2, y1],
                [x2, y2],
                [x1, y2],
                [x1, y1],
            ],
            dtype=float,
        )

        hsi_annotations.append({
            "modality": "HSI",
            "ink": ink,
            "raw_name": ink,
            "type": "detected_rectangle",
            "points": rect_points,
            "centroid": c["centroid"],
            "bbox": c["bbox"],
            "area": c["area"],
            "fill_frac": c["fill_frac"],
        })

    meta = {
        "png_path": str(hsi_ink_png_path),
        "image_size": (W, H),
        "image_rgb": img_rgb,
        "debug_masks": debug_masks,
    }

    return hsi_annotations, meta


def _extract_all_ink_annotations(
    he_xml_path,
    hsi_ink_png_path,
    keep_inks=("green", "red", "blue"),
    verbose=True,
):
    he_annotations, he_meta = _read_he_xml_ink_annotations(
        he_xml_path,
        keep_inks=keep_inks
    )

    hsi_annotations, hsi_meta = _detect_hsi_png_ink_annotations(
        hsi_ink_png_path,
        keep_inks=keep_inks
    )

    common_inks = sorted(
        set(a["ink"] for a in he_annotations) &
        set(a["ink"] for a in hsi_annotations)
    )

    annotations = {
        "HE": he_annotations,
        "HSI": hsi_annotations,
        "common_inks": common_inks,
        "meta": {
            "HE": he_meta,
            "HSI": {
                "png_path": hsi_meta["png_path"],
                "image_size": hsi_meta["image_size"],
            }
        }
    }

    if verbose:
        print("\n==============================")
        print("ANOTACIONES EXTRAÍDAS")
        print("==============================")
        print("HE meta:", annotations["meta"]["HE"])
        print("HSI meta:", annotations["meta"]["HSI"])

        print("\nH&E inks:")
        for a in he_annotations:
            print(
                f"  {a['ink']:>5s} | raw={a['raw_name']} | "
                f"centroid={np.round(a['centroid'], 2)} | bbox={a['bbox']}"
            )

        print("\nHSI inks:")
        for a in hsi_annotations:
            print(
                f"  {a['ink']:>5s} | "
                f"centroid={np.round(a['centroid'], 2)} | bbox={a['bbox']}"
            )

        print("\ncommon_inks:", common_inks)

    return annotations


# ============================================================
# 3. Pasar anotaciones a coordenadas del espécimen
# ============================================================

def _transform_annotations_to_specimen_space(
    image_rgb,
    mask_specimen,
    annotations_list,
    modality_name,
    annotation_scale=(1.0, 1.0),
    bbox_margin=20,
    show_plot=True,
    verbose=True,
):
    """
    Convierte anotaciones a:
        - coordenadas de imagen
        - coordenadas locales al crop del espécimen
        - coordenadas normalizadas del espécimen
    """
    image_rgb = np.asarray(image_rgb)
    mask = mask_specimen > 0

    img_h, img_w = image_rgb.shape[:2]

    if mask.shape[:2] != (img_h, img_w):
        raise ValueError(
            f"{modality_name}: imagen y máscara no tienen el mismo tamaño.\n"
            f"image_rgb: {image_rgb.shape[:2]}\n"
            f"mask:      {mask.shape[:2]}"
        )

    x1, y1, x2, y2 = _bbox_from_mask(mask, margin=bbox_margin)

    crop_rgb = image_rgb[y1:y2 + 1, x1:x2 + 1].copy()
    crop_mask = mask[y1:y2 + 1, x1:x2 + 1].copy()

    crop_h, crop_w = crop_rgb.shape[:2]

    sx, sy = annotation_scale

    transformed_annotations = []

    for ann in annotations_list:
        pts_original = np.asarray(ann["points"], dtype=float)
        centroid_original = np.asarray(ann["centroid"], dtype=float)

        pts_img = pts_original.copy()
        pts_img[:, 0] *= sx
        pts_img[:, 1] *= sy

        centroid_img = centroid_original.copy()
        centroid_img[0] *= sx
        centroid_img[1] *= sy

        pts_local = pts_img.copy()
        pts_local[:, 0] -= x1
        pts_local[:, 1] -= y1

        centroid_local = centroid_img.copy()
        centroid_local[0] -= x1
        centroid_local[1] -= y1

        pts_norm = pts_local.copy()
        pts_norm[:, 0] /= crop_w
        pts_norm[:, 1] /= crop_h

        centroid_norm = np.array(
            [
                centroid_local[0] / crop_w,
                centroid_local[1] / crop_h,
            ],
            dtype=float
        )

        cx = int(round(centroid_img[0]))
        cy = int(round(centroid_img[1]))

        if 0 <= cx < img_w and 0 <= cy < img_h:
            inside_mask = bool(mask[cy, cx])
        else:
            inside_mask = False

        transformed_annotations.append({
            **ann,
            "points_img": pts_img,
            "centroid_img": centroid_img,
            "points_local": pts_local,
            "centroid_local": centroid_local,
            "points_norm_specimen": pts_norm,
            "centroid_norm_specimen": centroid_norm,
            "centroid_inside_specimen": inside_mask,
        })

    result = {
        "modality": modality_name,
        "image_rgb": image_rgb,
        "mask": mask,
        "bbox": (x1, y1, x2, y2),
        "crop_rgb": crop_rgb,
        "crop_mask": crop_mask,
        "crop_size": (crop_w, crop_h),
        "image_size": (img_w, img_h),
        "annotation_scale": annotation_scale,
        "annotations": transformed_annotations,
    }

    if verbose:
        print("\n====================================")
        print(f"{modality_name} - coordenadas relativas al espécimen")
        print("====================================")
        print("image_size:", (img_w, img_h))
        print("specimen_bbox:", (x1, y1, x2, y2))
        print("crop_size:", (crop_w, crop_h))
        print("annotation_scale:", annotation_scale)

        for a in transformed_annotations:
            print(
                f"{modality_name} {a['ink']:>5s} | "
                f"centroid_img={np.round(a['centroid_img'], 2)} | "
                f"centroid_local={np.round(a['centroid_local'], 2)} | "
                f"centroid_norm={np.round(a['centroid_norm_specimen'], 4)} | "
                f"inside_mask={a['centroid_inside_specimen']}"
            )

    if show_plot:
        plt.figure(figsize=(8, 8))
        plt.imshow(crop_rgb)
        plt.imshow(crop_mask, cmap="gray", alpha=0.25)

        for a in transformed_annotations:
            color = _get_ink_color_mpl(a["ink"])

            plt.gca().add_patch(
                Polygon(
                    a["points_local"],
                    closed=True,
                    fill=False,
                    edgecolor=color,
                    linewidth=3
                )
            )

            cx, cy = a["centroid_local"]

            plt.scatter(
                [cx],
                [cy],
                c=color,
                s=80,
                edgecolors="black",
                zorder=10
            )

            plt.text(
                cx + 5,
                cy + 5,
                f"{a['ink']}\n{np.round(a['centroid_norm_specimen'], 3)}",
                color="black",
                fontsize=9,
                bbox=dict(facecolor=color, alpha=0.8, edgecolor="black")
            )

        plt.title(f"{modality_name}: crop espécimen + coordenadas normalizadas")
        plt.axis("off")
        plt.show()

    return result


# ============================================================
# 4. Orientación discreta: rotaciones/flips
# ============================================================

def _apply_orientation_image(img, k_rot90=0, flip_h=False):
    out = img.copy()

    if flip_h:
        out = np.fliplr(out)

    out = np.rot90(out, k=k_rot90)

    return out.copy()


def _apply_orientation_points(points, width, height, k_rot90=0, flip_h=False):
    pts = np.asarray(points, dtype=float).copy()

    current_w = width
    current_h = height

    if flip_h:
        pts[:, 0] = (current_w - 1) - pts[:, 0]

    for _ in range(k_rot90):
        x = pts[:, 0].copy()
        y = pts[:, 1].copy()

        pts[:, 0] = y
        pts[:, 1] = (current_w - 1) - x

        current_w, current_h = current_h, current_w

    return pts, current_w, current_h


def _get_orientation_candidates():
    candidates = []

    for flip_h in [False, True]:
        for k in [0, 1, 2, 3]:
            name = f"{'flipH_' if flip_h else ''}rot{k * 90}_ccw"
            candidates.append({
                "name": name,
                "k_rot90": k,
                "flip_h": flip_h,
            })

    return candidates


def _binary_edge(mask):
    mask_u8 = (mask > 0).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

    dil = cv2.dilate(mask_u8, kernel, iterations=1)
    ero = cv2.erode(mask_u8, kernel, iterations=1)

    return (dil - ero) > 0


def _symmetric_chamfer_distance(mask_a, mask_b):
    mask_a = mask_a > 0
    mask_b = mask_b > 0

    edge_a = _binary_edge(mask_a)
    edge_b = _binary_edge(mask_b)

    if edge_a.sum() < 5 or edge_b.sum() < 5:
        return np.inf

    h, w = mask_a.shape[:2]
    diag = np.sqrt(h ** 2 + w ** 2)

    inv_b = (~edge_b).astype(np.uint8) * 255
    inv_a = (~edge_a).astype(np.uint8) * 255

    dist_to_b = cv2.distanceTransform(inv_b, cv2.DIST_L2, 3)
    dist_to_a = cv2.distanceTransform(inv_a, cv2.DIST_L2, 3)

    d_ab = dist_to_b[edge_a].mean()
    d_ba = dist_to_a[edge_b].mean()

    return float((d_ab + d_ba) / 2.0 / diag)


def _evaluate_he_orientations_against_hsi(
    he_space,
    hsi_space,
    ink_weight=0.80,
    shape_weight=0.20,
    verbose=True,
):
    he_img = he_space["crop_rgb"]
    he_mask = he_space["crop_mask"]
    hsi_mask = hsi_space["crop_mask"]

    he_h, he_w = he_img.shape[:2]
    hsi_h, hsi_w = hsi_space["crop_rgb"].shape[:2]

    he_by_ink = {a["ink"]: a for a in he_space["annotations"]}
    hsi_by_ink = {a["ink"]: a for a in hsi_space["annotations"]}

    common_inks = sorted(set(he_by_ink.keys()) & set(hsi_by_ink.keys()))

    results = []

    for cand in _get_orientation_candidates():
        k = cand["k_rot90"]
        flip_h = cand["flip_h"]

        he_img_oriented = _apply_orientation_image(
            he_img,
            k_rot90=k,
            flip_h=flip_h,
        )

        he_mask_oriented = _apply_orientation_image(
            he_mask.astype(np.uint8),
            k_rot90=k,
            flip_h=flip_h,
        ) > 0

        oriented_annotations = {}

        for ink, ann in he_by_ink.items():
            pts_local = ann["points_local"]
            centroid_local = ann["centroid_local"].reshape(1, 2)

            pts_t, new_w, new_h = _apply_orientation_points(
                pts_local,
                width=he_w,
                height=he_h,
                k_rot90=k,
                flip_h=flip_h,
            )

            centroid_t, _, _ = _apply_orientation_points(
                centroid_local,
                width=he_w,
                height=he_h,
                k_rot90=k,
                flip_h=flip_h,
            )

            centroid_t = centroid_t[0]

            centroid_norm = np.array(
                [
                    centroid_t[0] / new_w,
                    centroid_t[1] / new_h,
                ],
                dtype=float
            )

            oriented_annotations[ink] = {
                "points_local_oriented": pts_t,
                "centroid_local_oriented": centroid_t,
                "centroid_norm_oriented": centroid_norm,
            }

        ink_errors = []

        for ink in common_inks:
            he_p = oriented_annotations[ink]["centroid_norm_oriented"]
            hsi_p = hsi_by_ink[ink]["centroid_norm_specimen"]
            ink_errors.append(np.linalg.norm(he_p - hsi_p))

        if len(ink_errors) > 0:
            ink_error = float(np.mean(ink_errors))
            total_ink_weight = ink_weight
            total_shape_weight = shape_weight
        else:
            ink_error = 0.0
            total_ink_weight = 0.0
            total_shape_weight = 1.0

        he_mask_resized = cv2.resize(
            he_mask_oriented.astype(np.uint8),
            (hsi_w, hsi_h),
            interpolation=cv2.INTER_NEAREST,
        ) > 0

        shape_error = _symmetric_chamfer_distance(
            he_mask_resized,
            hsi_mask,
        )

        score = total_ink_weight * ink_error + total_shape_weight * shape_error

        results.append({
            "name": cand["name"],
            "k_rot90": k,
            "flip_h": flip_h,
            "score": score,
            "ink_error": ink_error,
            "shape_error": shape_error,
            "he_img_oriented": he_img_oriented,
            "he_mask_oriented": he_mask_oriented,
            "he_mask_resized_to_hsi": he_mask_resized,
            "oriented_annotations": oriented_annotations,
        })

    results = sorted(results, key=lambda r: r["score"])

    if verbose:
        print("\n====================================")
        print("Ranking de orientaciones")
        print("====================================")
        print("Tintas comunes usadas:", common_inks)

        for i, r in enumerate(results):
            print(
                f"{i + 1:02d}. {r['name']:18s} | "
                f"score={r['score']:.4f} | "
                f"ink={r['ink_error']:.4f} | "
                f"shape={r['shape_error']:.4f}"
            )

            for ink in common_inks:
                he_p = r["oriented_annotations"][ink]["centroid_norm_oriented"]
                hsi_p = hsi_by_ink[ink]["centroid_norm_specimen"]

                print(
                    f"      {ink}: HE_oriented={np.round(he_p, 4)} "
                    f"HSI={np.round(hsi_p, 4)}"
                )

    return results


def _build_oriented_he_space_from_result(best_result, he_space):
    he_img_oriented = best_result["he_img_oriented"]
    he_mask_oriented = best_result["he_mask_oriented"]

    h_or, w_or = he_img_oriented.shape[:2]

    oriented_annotations = []

    for ann in he_space["annotations"]:
        ink = ann["ink"]

        if ink not in best_result["oriented_annotations"]:
            continue

        ann_or = best_result["oriented_annotations"][ink]

        points_local_oriented = ann_or["points_local_oriented"]
        centroid_local_oriented = ann_or["centroid_local_oriented"]
        centroid_norm_oriented = ann_or["centroid_norm_oriented"]

        oriented_annotations.append({
            **ann,
            "points_local": points_local_oriented,
            "centroid_local": centroid_local_oriented,
            "points_norm_specimen": points_local_oriented / np.array([w_or, h_or]),
            "centroid_norm_specimen": centroid_norm_oriented,
            "orientation_name": best_result["name"],
        })

    return {
        "modality": "HE_oriented",
        "orientation_name": best_result["name"],
        "k_rot90": best_result["k_rot90"],
        "flip_h": best_result["flip_h"],
        "crop_rgb": he_img_oriented,
        "crop_mask": he_mask_oriented,
        "annotations": oriented_annotations,
        "crop_size": (w_or, h_or),
        "score": best_result["score"],
        "ink_error": best_result["ink_error"],
        "shape_error": best_result["shape_error"],
    }


# ============================================================
# 5. Alineación inicial + refinamiento opcional por tintas
# ============================================================

def _build_initial_affine_from_masks(
    he_mask,
    hsi_mask,
    use_uniform_scale=True,
):
    he_bbox = _binary_bbox(he_mask)
    hsi_bbox = _binary_bbox(hsi_mask)

    he_center, he_size = _bbox_center_and_size(he_bbox)
    hsi_center, hsi_size = _bbox_center_and_size(hsi_bbox)

    sx = hsi_size[0] / he_size[0]
    sy = hsi_size[1] / he_size[1]

    if use_uniform_scale:
        s = 0.5 * (sx + sy)
        sx = s
        sy = s

    tx = hsi_center[0] - sx * he_center[0]
    ty = hsi_center[1] - sy * he_center[1]

    M = np.array(
        [
            [sx, 0.0, tx],
            [0.0, sy, ty],
        ],
        dtype=np.float32
    )

    info = {
        "he_bbox": he_bbox,
        "hsi_bbox": hsi_bbox,
        "he_center": he_center,
        "hsi_center": hsi_center,
        "he_size": he_size,
        "hsi_size": hsi_size,
        "sx": sx,
        "sy": sy,
        "tx": tx,
        "ty": ty,
        "M": M,
        "use_uniform_scale": use_uniform_scale,
    }

    return M, info


def _estimate_similarity_transform_2d(src_pts, dst_pts):
    src = np.asarray(src_pts, dtype=np.float64)
    dst = np.asarray(dst_pts, dtype=np.float64)

    if src.shape[0] < 2:
        raise ValueError("Se necesitan al menos 2 puntos para similarity transform.")

    mu_src = src.mean(axis=0)
    mu_dst = dst.mean(axis=0)

    X = src - mu_src
    Y = dst - mu_dst

    var_src = np.mean(np.sum(X ** 2, axis=1))

    if var_src < 1e-8:
        raise ValueError("Puntos fuente degenerados.")

    cov = (Y.T @ X) / src.shape[0]

    U, S, Vt = np.linalg.svd(cov)

    D = np.eye(2)

    if np.linalg.det(U @ Vt) < 0:
        D[-1, -1] = -1

    R = U @ D @ Vt
    scale = np.trace(np.diag(S) @ D) / var_src
    t = mu_dst - scale * R @ mu_src

    M = np.array(
        [
            [scale * R[0, 0], scale * R[0, 1], t[0]],
            [scale * R[1, 0], scale * R[1, 1], t[1]],
        ],
        dtype=np.float32
    )

    return M


def _refine_affine_by_common_inks(
    M_initial,
    he_oriented_space,
    hsi_space,
    correction_strength=0.75,
):
    """
    Refinamiento general:
    - 0 tintas comunes: no cambia nada.
    - 1 tinta común: traslación.
    - 2/3 tintas comunes: similarity transform usando todas.
    """
    he_by_ink = {a["ink"]: a for a in he_oriented_space["annotations"]}
    hsi_by_ink = {a["ink"]: a for a in hsi_space["annotations"]}

    common_inks = sorted(set(he_by_ink.keys()) & set(hsi_by_ink.keys()))

    if len(common_inks) == 0:
        return M_initial, {
            "mode": "no_common_inks",
            "common_inks": common_inks,
            "M_residual": np.array([[1, 0, 0], [0, 1, 0]], dtype=np.float32),
        }

    he_initial_pts = []
    hsi_target_pts = []

    for ink in common_inks:
        he_local = he_by_ink[ink]["centroid_local"]
        hsi_target = hsi_by_ink[ink]["centroid_local"]

        he_initial = _transform_points_affine(he_local, M_initial)[0]

        he_initial_pts.append(he_initial)
        hsi_target_pts.append(hsi_target)

    he_initial_pts = np.asarray(he_initial_pts, dtype=np.float64)
    hsi_target_pts = np.asarray(hsi_target_pts, dtype=np.float64)

    if len(common_inks) == 1:
        delta = hsi_target_pts[0] - he_initial_pts[0]

        M_residual = np.array(
            [
                [1.0, 0.0, delta[0]],
                [0.0, 1.0, delta[1]],
            ],
            dtype=np.float32
        )

        mode = "translation_1_ink"

    else:
        M_residual = _estimate_similarity_transform_2d(
            he_initial_pts,
            hsi_target_pts,
        )
        mode = f"similarity_{len(common_inks)}_inks"

    I3 = np.eye(3, dtype=np.float64)
    M_residual_3 = _affine_2x3_to_3x3(M_residual)

    # Corrección suave, para no confiar al 100% en anotaciones aproximadas.
    M_residual_soft_3 = I3 + correction_strength * (M_residual_3 - I3)

    M_initial_3 = _affine_2x3_to_3x3(M_initial)

    M_refined_3 = M_residual_soft_3 @ M_initial_3
    M_refined = _affine_3x3_to_2x3(M_refined_3)

    info = {
        "mode": mode,
        "common_inks": common_inks,
        "M_residual": M_residual,
        "he_initial_pts": he_initial_pts,
        "hsi_target_pts": hsi_target_pts,
        "correction_strength": correction_strength,
    }

    return M_refined, info


def _warp_he_to_hsi_space(
    he_oriented_space,
    hsi_space,
    M,
):
    he_rgb = he_oriented_space["crop_rgb"]
    he_mask = he_oriented_space["crop_mask"] > 0

    hsi_rgb = hsi_space["crop_rgb"]
    hsi_h, hsi_w = hsi_rgb.shape[:2]

    he_rgb_warped = cv2.warpAffine(
        he_rgb,
        M,
        (hsi_w, hsi_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )

    he_mask_warped_u8 = cv2.warpAffine(
        (he_mask.astype(np.uint8) * 255),
        M,
        (hsi_w, hsi_h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    he_mask_warped = he_mask_warped_u8 > 0

    warped_annotations = []

    for ann in he_oriented_space["annotations"]:
        pts = np.asarray(ann["points_local"], dtype=float)
        centroid = np.asarray(ann["centroid_local"], dtype=float)

        pts_warped = _transform_points_affine(pts, M)
        centroid_warped = _transform_points_affine(centroid, M)[0]

        warped_annotations.append({
            **ann,
            "points_hsi_crop": pts_warped,
            "centroid_hsi_crop": centroid_warped,
            "centroid_norm_hsi_crop": np.array(
                [
                    centroid_warped[0] / hsi_w,
                    centroid_warped[1] / hsi_h,
                ],
                dtype=float
            ),
        })

    return he_rgb_warped, he_mask_warped, warped_annotations


# ============================================================
# 6. Crear overlays de salida
# ============================================================

def _make_overlay_rgb(hsi_rgb, he_rgb_warped, he_mask_warped, alpha=0.45):
    overlay = hsi_rgb.copy().astype(np.float32)
    he_float = he_rgb_warped.astype(np.float32)

    m = he_mask_warped > 0
    overlay[m] = (1 - alpha) * overlay[m] + alpha * he_float[m]

    return np.clip(overlay, 0, 255).astype(np.uint8)


def _draw_contours_and_inks_rgb(
    hsi_rgb,
    hsi_mask,
    he_mask_warped,
    hsi_annotations,
    he_annotations_warped,
):
    canvas = hsi_rgb.copy()

    # Contorno HSI: amarillo
    contours_hsi, _ = cv2.findContours(
        (hsi_mask.astype(np.uint8) * 255),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    cv2.drawContours(canvas, contours_hsi, -1, (255, 255, 0), 2)

    # Contorno H&E: cian
    contours_he, _ = cv2.findContours(
        (he_mask_warped.astype(np.uint8) * 255),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    cv2.drawContours(canvas, contours_he, -1, (0, 255, 255), 2)

    # Tintas HSI: cuadrados
    for ann in hsi_annotations:
        color = _get_ink_color_rgb(ann["ink"])
        cx, cy = ann["centroid_local"]
        cx = int(round(cx))
        cy = int(round(cy))

        cv2.rectangle(canvas, (cx - 5, cy - 5), (cx + 5, cy + 5), color, -1)
        cv2.putText(
            canvas,
            f"HSI {ann['ink']}",
            (cx + 8, cy - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            color,
            1,
            cv2.LINE_AA,
        )

    # Tintas H&E: círculos
    for ann in he_annotations_warped:
        color = _get_ink_color_rgb(ann["ink"])
        cx, cy = ann["centroid_hsi_crop"]
        cx = int(round(cx))
        cy = int(round(cy))

        cv2.circle(canvas, (cx, cy), 6, color, -1)
        cv2.circle(canvas, (cx, cy), 6, (0, 0, 0), 1)

        cv2.putText(
            canvas,
            f"HE {ann['ink']}",
            (cx + 8, cy + 14),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            color,
            1,
            cv2.LINE_AA,
        )

    return canvas


# ============================================================
# 7. FUNCIÓN PRINCIPAL
# ============================================================

def orient_he_hsi_specimens(
    slide_path,
    he_xml_path,
    hdr_path,
    hsi_ink_png_path,

    # Segmentación H&E
    he_max_dim=1800,
    he_segmentation_kwargs=None,

    # Segmentación HSI
    hsi_segmentation_kwargs=None,

    # Anotaciones
    keep_inks=("green", "red", "blue"),

    # Coordenadas
    bbox_margin=20,

    # Orientación discreta
    ink_weight=0.80,
    shape_weight=0.20,

    # Alineación inicial
    use_uniform_scale=True,

    # Refinamiento por tintas
    refine_with_inks=True,
    ink_correction_strength=0.75,

    # Visualización
    show_plots=True,
    show_segmentation_plots=True,
    debug_segmentation=False,
    verbose=True,
    overlay_alpha=0.45,
):
    """
    Pipeline completo:
    1) Extrae tintas H&E desde XML.
    2) Detecta tintas HSI desde PNG.
    3) Extrae espécimen H&E desde MRXS.
    4) Extrae espécimen HSI desde HDR.
    5) Convierte tintas a coordenadas relativas al espécimen.
    6) Prueba orientaciones discretas de H&E.
    7) Elige la mejor orientación.
    8) Alinea H&E con HSI por máscara.
    9) Opcionalmente refina con todas las tintas comunes.
    10) Devuelve imágenes y metadatos.
    """

    slide_path = Path(slide_path)
    he_xml_path = Path(he_xml_path)
    hdr_path = Path(hdr_path)
    hsi_ink_png_path = Path(hsi_ink_png_path)

    he_segmentation_kwargs = he_segmentation_kwargs or {}
    hsi_segmentation_kwargs = hsi_segmentation_kwargs or {}

    # ------------------------------------------------------------
    # 1. Extraer anotaciones
    # ------------------------------------------------------------
    annotations = _extract_all_ink_annotations(
        he_xml_path=he_xml_path,
        hsi_ink_png_path=hsi_ink_png_path,
        keep_inks=keep_inks,
        verbose=verbose,
    )

    # ------------------------------------------------------------
    # 2. Extraer espécimen H&E
    # ------------------------------------------------------------
    if verbose:
        print("\n==============================")
        print("EXTRAYENDO ESPÉCIMEN H&E")
        print("==============================")

    he_tissue_rgb, he_mask, he_info = extract_he_tissue_contour_image(
        slide_path=slide_path,
        max_dim=he_max_dim,
        show_original=show_segmentation_plots,
        show_result=show_segmentation_plots,
        debug=debug_segmentation,
        return_mask=True,
        return_info=True,
        **he_segmentation_kwargs,
    )

    if verbose:
        print("he_info:")
        for k, v in he_info.items():
            print(f"  {k}: {v}")
        print("he_tissue_rgb.shape:", he_tissue_rgb.shape)
        print("he_mask.shape:", he_mask.shape)

    # ------------------------------------------------------------
    # 3. Extraer espécimen HSI
    # ------------------------------------------------------------
    if verbose:
        print("\n==============================")
        print("EXTRAYENDO ESPÉCIMEN HSI")
        print("==============================")

    hsi_specimen_rgb_full, hsi_mask_full = extract_specimen_only_from_hsi_path(
        hdr_path=str(hdr_path),
        show_original=show_segmentation_plots,
        show_result=show_segmentation_plots,
        return_mask=True,
        **hsi_segmentation_kwargs,
    )

    if verbose:
        print("hsi_specimen_rgb_full.shape:", hsi_specimen_rgb_full.shape)
        print("hsi_mask_full.shape:", hsi_mask_full.shape)

    # ------------------------------------------------------------
    # 4. Escalas de anotaciones
    # ------------------------------------------------------------
    xml_w, xml_h = annotations["meta"]["HE"]["coord_size_used"]
    he_w, he_h = he_info["read_dimensions"]

    he_scale_x = he_w / xml_w
    he_scale_y = he_h / xml_h

    png_w, png_h = annotations["meta"]["HSI"]["image_size"]
    hsi_h, hsi_w = hsi_specimen_rgb_full.shape[:2]

    hsi_scale_x = hsi_w / png_w
    hsi_scale_y = hsi_h / png_h

    if verbose:
        print("\n==============================")
        print("ESCALAS DE ANOTACIONES")
        print("==============================")
        print("H&E scale:", (he_scale_x, he_scale_y))
        print("HSI scale:", (hsi_scale_x, hsi_scale_y))

    # ------------------------------------------------------------
    # 5. Pasar anotaciones a coordenadas de espécimen
    # ------------------------------------------------------------
    he_space = _transform_annotations_to_specimen_space(
        image_rgb=he_tissue_rgb,
        mask_specimen=he_mask,
        annotations_list=annotations["HE"],
        modality_name="HE",
        annotation_scale=(he_scale_x, he_scale_y),
        bbox_margin=bbox_margin,
        show_plot=show_plots,
        verbose=verbose,
    )

    hsi_space = _transform_annotations_to_specimen_space(
        image_rgb=hsi_specimen_rgb_full,
        mask_specimen=hsi_mask_full,
        annotations_list=annotations["HSI"],
        modality_name="HSI",
        annotation_scale=(hsi_scale_x, hsi_scale_y),
        bbox_margin=bbox_margin,
        show_plot=show_plots,
        verbose=verbose,
    )

    # ------------------------------------------------------------
    # 6. Probar orientaciones discretas
    # ------------------------------------------------------------
    orientation_results = _evaluate_he_orientations_against_hsi(
        he_space=he_space,
        hsi_space=hsi_space,
        ink_weight=ink_weight,
        shape_weight=shape_weight,
        verbose=verbose,
    )

    best_orientation = orientation_results[0]

    he_oriented_space = _build_oriented_he_space_from_result(
        best_result=best_orientation,
        he_space=he_space,
    )

    if verbose:
        print("\n==============================")
        print("ORIENTACIÓN ELEGIDA")
        print("==============================")
        print("orientation_name:", he_oriented_space["orientation_name"])
        print("score:", he_oriented_space["score"])
        print("ink_error:", he_oriented_space["ink_error"])
        print("shape_error:", he_oriented_space["shape_error"])

    # ------------------------------------------------------------
    # 7. Alineación inicial por máscaras
    # ------------------------------------------------------------
    M_initial, initial_affine_info = _build_initial_affine_from_masks(
        he_mask=he_oriented_space["crop_mask"],
        hsi_mask=hsi_space["crop_mask"],
        use_uniform_scale=use_uniform_scale,
    )

    if verbose:
        print("\n==============================")
        print("AFFINE INICIAL H&E -> HSI")
        print("==============================")
        print("use_uniform_scale:", use_uniform_scale)
        print("HE bbox:", initial_affine_info["he_bbox"])
        print("HSI bbox:", initial_affine_info["hsi_bbox"])
        print("sx:", initial_affine_info["sx"])
        print("sy:", initial_affine_info["sy"])
        print("tx:", initial_affine_info["tx"])
        print("ty:", initial_affine_info["ty"])
        print("M_initial:\n", M_initial)

    # ------------------------------------------------------------
    # 8. Refinamiento opcional con todas las tintas comunes
    # ------------------------------------------------------------
    if refine_with_inks:
        M_final, refine_info = _refine_affine_by_common_inks(
            M_initial=M_initial,
            he_oriented_space=he_oriented_space,
            hsi_space=hsi_space,
            correction_strength=ink_correction_strength,
        )
    else:
        M_final = M_initial.copy()
        refine_info = {
            "mode": "disabled",
            "common_inks": annotations["common_inks"],
            "correction_strength": 0.0,
        }

    if verbose:
        print("\n==============================")
        print("REFINAMIENTO POR TINTAS")
        print("==============================")
        print("refine_with_inks:", refine_with_inks)
        print("mode:", refine_info["mode"])
        print("common_inks:", refine_info["common_inks"])
        print("correction_strength:", refine_info.get("correction_strength"))
        print("M_final:\n", M_final)

    # ------------------------------------------------------------
    # 9. Warpear H&E final al espacio HSI
    # ------------------------------------------------------------
    he_aligned_rgb, he_aligned_mask, he_aligned_annotations = _warp_he_to_hsi_space(
        he_oriented_space=he_oriented_space,
        hsi_space=hsi_space,
        M=M_final,
    )

    # ------------------------------------------------------------
    # 10. Crear overlays de salida
    # ------------------------------------------------------------
    overlay_rgb = _make_overlay_rgb(
        hsi_rgb=hsi_space["crop_rgb"],
        he_rgb_warped=he_aligned_rgb,
        he_mask_warped=he_aligned_mask,
        alpha=overlay_alpha,
    )

    contours_inks_rgb = _draw_contours_and_inks_rgb(
        hsi_rgb=hsi_space["crop_rgb"],
        hsi_mask=hsi_space["crop_mask"],
        he_mask_warped=he_aligned_mask,
        hsi_annotations=hsi_space["annotations"],
        he_annotations_warped=he_aligned_annotations,
    )

    # ------------------------------------------------------------
    # 11. Plots finales
    # ------------------------------------------------------------
    if show_plots:
        fig, axes = plt.subplots(1, 5, figsize=(26, 6))

        axes[0].imshow(hsi_space["crop_rgb"])
        axes[0].imshow(hsi_space["crop_mask"], cmap="gray", alpha=0.25)
        axes[0].set_title("HSI espécimen")
        axes[0].axis("off")

        axes[1].imshow(he_oriented_space["crop_rgb"])
        axes[1].imshow(he_oriented_space["crop_mask"], cmap="gray", alpha=0.25)
        axes[1].set_title(f"H&E espécimen orientada\n{he_oriented_space['orientation_name']}")
        axes[1].axis("off")

        axes[2].imshow(he_aligned_rgb)
        axes[2].imshow(he_aligned_mask, cmap="gray", alpha=0.25)
        axes[2].set_title("H&E orientada + alineada")
        axes[2].axis("off")

        axes[3].imshow(overlay_rgb)
        axes[3].set_title("Overlay HSI + H&E")
        axes[3].axis("off")

        axes[4].imshow(contours_inks_rgb)
        axes[4].set_title("Contornos + tintas\namarillo=HSI | cian=H&E")
        axes[4].axis("off")

        plt.tight_layout()
        plt.show()

    # ------------------------------------------------------------
    # 12. Confianza orientativa
    # ------------------------------------------------------------
    n_common = len(annotations["common_inks"])

    if n_common >= 2:
        orientation_confidence = "high"
    elif n_common == 1:
        orientation_confidence = "medium"
    else:
        orientation_confidence = "low"

    result = {
        # Entradas
        "paths": {
            "slide_path": str(slide_path),
            "he_xml_path": str(he_xml_path),
            "hdr_path": str(hdr_path),
            "hsi_ink_png_path": str(hsi_ink_png_path),
        },

        # Anotaciones
        "annotations": annotations,

        # Espacios intermedios
        "he_space": he_space,
        "hsi_space": hsi_space,
        "he_oriented_space": he_oriented_space,

        # Orientación
        "orientation_results": orientation_results,
        "best_orientation": best_orientation,
        "orientation_confidence": orientation_confidence,

        # Transformaciones
        "M_initial_he_to_hsi": M_initial,
        "initial_affine_info": initial_affine_info,
        "M_final_he_to_hsi": M_final,
        "refine_info": refine_info,

        # Salidas principales
        "hsi_specimen_rgb": hsi_space["crop_rgb"],
        "hsi_specimen_mask": hsi_space["crop_mask"],

        "he_oriented_rgb": he_oriented_space["crop_rgb"],
        "he_oriented_mask": he_oriented_space["crop_mask"],

        "he_aligned_rgb": he_aligned_rgb,
        "he_aligned_mask": he_aligned_mask,
        "he_aligned_annotations": he_aligned_annotations,

        "overlay_rgb": overlay_rgb,
        "contours_inks_rgb": contours_inks_rgb,

        # Info
        "he_info": he_info,
    }

    return result

## Pruebas

In [ ]:
slide_path = r"Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs"
he_xml_path = r"Datos\SB013\SB013\H&E_python_EDU_creado\H&E.xml"

hdr_path = r"Datos\SB013\SB013\SB013_001\SB013_nrm.hdr"
hsi_ink_png_path = r"Datos\SB013\SB013\SB013_ink_annotation.png"

result = orient_he_hsi_specimens(
    slide_path=slide_path,
    he_xml_path=he_xml_path,
    hdr_path=hdr_path,
    hsi_ink_png_path=hsi_ink_png_path,

    show_plots=True,
    show_segmentation_plots=True,
    debug_segmentation=False,
    verbose=True,

    refine_with_inks=True,
    ink_correction_strength=0.75,
    use_uniform_scale=True,
)

Salidas principales

In [ ]:
hsi_specimen_rgb = result["hsi_specimen_rgb"]
he_oriented_rgb = result["he_oriented_rgb"]
he_aligned_rgb = result["he_aligned_rgb"]
overlay_rgb = result["overlay_rgb"]
contours_inks_rgb = result["contours_inks_rgb"]

Plots

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(result["hsi_specimen_rgb"])
plt.title("HSI espécimen")
plt.axis("off")
plt.show()

plt.figure(figsize=(8, 6))
plt.imshow(result["he_oriented_rgb"])
plt.title("H&E espécimen orientada")
plt.axis("off")
plt.show()

plt.figure(figsize=(8, 6))
plt.imshow(result["overlay_rgb"])
plt.title("Overlay HSI + H&E")
plt.axis("off")
plt.show()

plt.figure(figsize=(8, 6))
plt.imshow(result["contours_inks_rgb"])
plt.title("Contornos + tintas")
plt.axis("off")
plt.show()